In [1]:
import warnings
warnings.filterwarnings('ignore')
import teehr.fetching.nwm.retrospective_points as teehr_retro
import teehr.fetching.nwm.nwm_points as teehr_operational
from datetime import datetime
import pandas as pd
import time
import json
import os
os.environ["FSSPEC_HTTP_TIMEOUT"] = "300"

In [2]:
def get_reach_list_for_huc(huc):
    with open(f'data/huc_{huc}_geometries.geojson') as f:
        data = json.load(f)
        reach_id_list = [feature['id'] for feature in data['features']]
    return reach_id_list

huc_0305010505_reach_id_list = get_reach_list_for_huc("0305010505")
huc_030501050505_reach_id_list = get_reach_list_for_huc("030501050505")

print(f"Number of reaches in HUC 0305010505: {len(huc_0305010505_reach_id_list)}")
print(f"Number of reaches in HUC 030501050505: {len(huc_030501050505_reach_id_list)}")

Number of reaches in HUC 0305010505: 115
Number of reaches in HUC 030501050505: 10


In [3]:
reach_mapper_dict = {
    "HUC-0305010505":  huc_0305010505_reach_id_list,
    "HUC-030501050505": huc_030501050505_reach_id_list,
    "REACH-12034579": [12034579,],
}

In [4]:
def get_retro_via_teehr(case, start_date, end_date, reach_id_list, variable, label):
    start = time.perf_counter()
    teehr_retro.nwm_retro_to_parquet(nwm_version="nwm30", 
                                    variable_name=variable,
                                    start_date=start_date, 
                                    end_date=end_date, 
                                    location_ids=reach_id_list,
                                    output_parquet_dir=f'data/teehr/retro_{case}/{label}')
    output_parquet_dir = f'data/teehr/retro_{case}/{label}/{start_date.strftime("%Y%m%d")}_{end_date.strftime("%Y%m%d")}.parquet'
    retro_teehr = pd.read_parquet(output_parquet_dir)
    print("Data dimension:", retro_teehr.shape)
    end = time.perf_counter()
    elapsed = end - start
    return elapsed

In [5]:
def retro_case_run(case, end_date, var):
    with open(f'results/teehr/retro_{case}.json', "w") as f:
        for label, reach_id_list in reach_mapper_dict.items():
            elapsed = get_retro_via_teehr(
                case = case,
                start_date = datetime(2022, 1, 1, 0, 0, 0), 
                end_date = end_date,
                reach_id_list = reach_id_list, 
                variable = var,
                label = label)
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("_" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n")   

In [6]:
# Retro Case 1: streamflow + 1 week
retro_case_run("case01", datetime(2022, 1, 8, 0, 0, 0), "streamflow")

Data dimension: (19435, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 16.861793832998956}
________________________________________
Data dimension: (1690, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 13.733983666999848}
________________________________________
Data dimension: (169, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 12.639703957989695}
________________________________________


In [7]:
# Retro Case 2: velocity + 1 week
retro_case_run("case02", datetime(2022, 1, 8, 0, 0, 0), "velocity")

Data dimension: (19435, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 12.82292629100266}
________________________________________
Data dimension: (1690, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 10.96249545799219}
________________________________________
Data dimension: (169, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 13.247927749995142}
________________________________________


In [8]:
# Retro Case 3: streamflow + 1 month
retro_case_run("case03", datetime(2022, 2, 1, 0, 0, 0), "streamflow")

Data dimension: (85675, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 15.984503707994008}
________________________________________
Data dimension: (7450, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 15.077170833013952}
________________________________________
Data dimension: (745, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 14.490288125001825}
________________________________________


In [9]:
# Retro Case 4: velocity + 1 month
retro_case_run("case04", datetime(2022, 2, 1, 0, 0, 0), "velocity")

Data dimension: (85675, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 11.747017290996155}
________________________________________
Data dimension: (7450, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 11.929015625006286}
________________________________________
Data dimension: (745, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 11.333617707990925}
________________________________________


In [12]:
def get_operational_via_teehr(config, case, start_date, end_date, reach_id_list, variable, data_source, label, output_type="channel_rt"):
    start = time.perf_counter()
    teehr_operational.nwm_to_parquet(configuration = config,
                                     output_type  = output_type,
                                     variable_name=variable,
                                     nwm_version = "nwm30",
                                     start_date=start_date,
                                     end_date=end_date,
                                     data_source=data_source,
                                     kerchunk_method="auto",
                                     location_ids=reach_id_list,
                                     json_dir=f'data/teehr/{config}_{case}/json',
                                     output_parquet_dir=f'data/teehr/{config}_{case}/{label}')
    output_parquet_dir = f'data/teehr/{config}_{case}/{label}'
    ops_data = pd.read_parquet(output_parquet_dir)
    print("Data dimension:", ops_data.shape)
    end = time.perf_counter()
    elapsed = end - start
    return elapsed

In [13]:
def analysis_case_run(case, end_date, var, data_source):
    with open(f'results/teehr/analysis_{case}.json', "w") as f:
        for label, reach_id_list in reach_mapper_dict.items():
            elapsed = get_operational_via_teehr(config="analysis_assim", 
                                                case = case,
                                                start_date = datetime(2024, 1, 1, 0, 0, 0), 
                                                end_date = end_date,
                                                reach_id_list = reach_id_list,
                                                variable = var,
                                                data_source = data_source,
                                                label = label)
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("-" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n")

In [14]:
# Analysis Case 1: streamflow + 1 week + GCS
analysis_case_run("case_01", datetime(2024, 1, 8, 0, 0, 0), "streamflow", "GCS")

Data dimension: (19435, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 194.79409412499808}
----------------------------------------
Data dimension: (1690, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 177.8720317910047}
----------------------------------------
Data dimension: (169, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 219.26621495799918}
----------------------------------------


In [15]:
# Analysis Case 2: velocity + 1 week + GCS
analysis_case_run("case_02", datetime(2024, 1, 8, 0, 0, 0), "velocity", "GCS")

Data dimension: (19435, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 196.83634204200644}
----------------------------------------
Data dimension: (1690, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 168.78343441701145}
----------------------------------------
Data dimension: (169, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 162.20431283400103}
----------------------------------------


In [16]:
# Analysis Case 3: streamflow + 1 month + GCS
analysis_case_run("case_03", datetime(2024, 2, 1, 0, 0, 0), "streamflow", "GCS")

Data dimension: (85675, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 799.4732180829888}
----------------------------------------
Data dimension: (7450, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 766.0932734160015}
----------------------------------------
Data dimension: (745, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 706.2621393750014}
----------------------------------------


In [18]:
def srf_case_run(case, var):
    with open(f'results/teehr/short_range_{case}.json', "w") as f:
        for label, reach_id_list in reach_mapper_dict.items():
            elapsed = get_operational_via_teehr(config="short_range", 
                                                case = case,
                                                start_date = datetime(2024, 1, 1, 0, 0, 0), 
                                                end_date = datetime(2024, 1, 1, 0, 0, 0),
                                                reach_id_list = reach_id_list,
                                                variable = var,
                                                data_source = 'GCS',
                                                label = label)
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("-" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n") 

In [19]:
# Short-range Case 1: streamflow + GCS
srf_case_run("case_01", 'streamflow')

Data dimension: (2070, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 6.128896499983966}
----------------------------------------
Data dimension: (180, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 5.346369000006234}
----------------------------------------
Data dimension: (18, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 5.228127250011312}
----------------------------------------


In [20]:
# Short-range Case 2: velocity + GCS
srf_case_run("case_02", 'velocity')

Data dimension: (2070, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 5.86725108299288}
----------------------------------------
Data dimension: (180, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 5.547327250009403}
----------------------------------------
Data dimension: (18, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 5.59688808399369}
----------------------------------------


In [21]:
def mrf_case_run(case, end_date, var, data_source, member):
    config_mapping = {1: "medium_range_mem1", 2: "medium_range_mem2", 3: "medium_range_mem3", 4: "medium_range_mem4", 5: "medium_range_mem5",
                      6: "medium_range_mem6"}
    with open(f'results/teehr/medium_range_{case}.json', "w") as f:
        for label, reach_id_list in reach_mapper_dict.items():
            elapsed = get_operational_via_teehr(config=config_mapping[member], 
                                                case = case,
                                                start_date = datetime(2024, 1, 1, 0, 0, 0), 
                                                end_date = end_date,
                                                reach_id_list = reach_id_list,
                                                variable = var,
                                                data_source = data_source,
                                                label = label,
                                                output_type = f"channel_rt_{member}")
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("-" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n") 

In [22]:
# MRF Case 1: streamflow + GCS
mrf_case_run("case_01", datetime(2024, 1, 1, 0, 0, 0), 'streamflow', 'GCS', member=1)

Data dimension: (27600, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 61.0030058749835}
----------------------------------------
Data dimension: (2400, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 60.75286441697972}
----------------------------------------
Data dimension: (240, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 58.95184945798246}
----------------------------------------


In [23]:
# MRF Case 2: velocity + GCS
mrf_case_run("case_02", datetime(2024, 1, 1, 0, 0, 0), 'velocity', 'GCS', member=1)

Data dimension: (27600, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 63.2019592919969}
----------------------------------------
Data dimension: (2400, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 62.319831125001656}
----------------------------------------
Data dimension: (240, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 62.17194883301272}
----------------------------------------


In [24]:
def lrf_case_run(case, end_date, var, data_source, member):
    config_mapping = {1: "long_range_mem1", 2: "long_range_mem2", 3: "long_range_mem3", 4: "long_range_mem4"}
    with open(f'results/teehr/long_range_{case}.json', "w") as f:
        for label, reach_id_list in reach_mapper_dict.items():
            elapsed = get_operational_via_teehr(config=config_mapping[member], 
                                                case = case,
                                                start_date = datetime(2024, 1, 1, 0, 0, 0), 
                                                end_date = end_date,
                                                reach_id_list = reach_id_list,
                                                variable = var,
                                                data_source = data_source,
                                                label = label,
                                                output_type = f"channel_rt_{member}")
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("-" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n") 

In [25]:
# LRF Case 1: streamflow + GCS
lrf_case_run("case_01", datetime(2024, 1, 1, 0, 0, 0), 'streamflow', 'GCS', member=1)

Data dimension: (13800, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 30.987961457984056}
----------------------------------------
Data dimension: (1200, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 30.495905166026205}
----------------------------------------
Data dimension: (120, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 30.360655459022382}
----------------------------------------


In [26]:
# LRF Case 2: velocity + GCS
lrf_case_run("case_02", datetime(2024, 1, 1, 0, 0, 0), 'velocity', 'GCS', member=1)

Data dimension: (13800, 8)
{'label': 'HUC-0305010505', 'elapsed_time_sec': 31.608876333018998}
----------------------------------------
Data dimension: (1200, 8)
{'label': 'HUC-030501050505', 'elapsed_time_sec': 31.815577125002164}
----------------------------------------
Data dimension: (120, 8)
{'label': 'REACH-12034579', 'elapsed_time_sec': 31.839078667020658}
----------------------------------------
